Software Name: ZeroShotACD-Supplemental-Material 

SPDX-FileCopyrightText: Copyright (c) 2025 Orange SA

SPDX-License-Identifier: MIT


This software is distributed under the MIT license,
the text of which is available at https://opensource.org/license/MIT/
or see the "LICENSE" file for more details.


Authors: See CONTRIBUTORS.txt

Software description: The supplemental material for the paper From Zero-Shot to Reward-Aware: Evaluating Prompting and Memory in LLM-Based Cyber Defense Agents".

# CybORG Install

In [ ]:
!git clone https://github.com/alan-turing-institute/CybORG_plus_plus.git

In [ ]:
!sed -i '250i \        proto_vector.append(success.value)\n' CybORG_plus_plus/Debugged_CybORG/CybORG/CybORG/Agents/Wrappers/BlueTableWrapper.py

In [ ]:
!sed -i '114d' CybORG_plus_plus/Debugged_CybORG/CybORG/CybORG/Agents/Wrappers/TrueTableWrapper.py

In [ ]:
!pip install -e CybORG_plus_plus/Debugged_CybORG/CybORG
!pip install paramiko

In [ ]:
import os
import sys
# Replace with absolute path toward CybORG++ folder
sys.path.insert(0, os.path.abspath('/content/CybORG_plus_plus/Debugged_CybORG/CybORG/'))

# Cardiff Install

In [ ]:
!git clone https://github.com/john-cardiff/-cyborg-cage-2.git cardiff

In [ ]:
%cd cardiff

In [ ]:
!sed -i "s/obs, reward, done, info = self.env.step(action=action)/obs, reward, done, truncated, info = self.env.step(action=action)/g" Wrappers/ChallengeWrapper2.py

In [ ]:
!sed -i '28s|.*|                self.agent = self.load_bline()|' Agents/MainAgent.py

In [ ]:
!sed -i '32s|.*|                self.agent = self.load_bline()|' Agents/MainAgent.py

In [ ]:
! rm Models/meander/model.pth

# Stats

In [ ]:
import numpy as np
from scipy import stats

In [ ]:
def interquartile_mean(x):
    x = np.asarray(x)
    q1 = np.percentile(x, 25)
    q3 = np.percentile(x, 75)
    mask = (x >= q1) & (x <= q3)
    return np.mean(x[mask])

In [ ]:
def get_iqm_with_bci(rewards):
    data = np.array(rewards)

    # Compute IQM
    iqm = interquartile_mean(data)

    # Compute bootstrap confidence interval for IQM
    bci = stats.bootstrap(
      (data,),
      interquartile_mean,
      confidence_level=0.95,
      method="percentile",
    )

    min_r = data.min()
    max_r = data.max()

    return iqm, bci, min_r, max_r

# Run

In [ ]:
import time
import random
import inspect
from statistics import mean, stdev

In [ ]:
from Agents.MainAgent import MainAgent
from CybORG import CybORG, CYBORG_VERSION
from Wrappers.ChallengeWrapper2 import ChallengeWrapper2
from CybORG.Agents import B_lineAgent, SleepAgent, GreenAgent
from CybORG.Agents.SimpleAgents.Meander import RedMeanderAgent

In [ ]:
MAX_EPS = 100
agent_name = 'Blue'
random.seed(42)

In [ ]:
def wrap(env):
    return ChallengeWrapper2(env=env, agent_name='Blue')

In [ ]:
cyborg_version = CYBORG_VERSION
scenario = 'Scenario2'

lines = inspect.getsource(wrap)
wrap_line = lines.split('\n')[1].split('return ')[1]

# Change this line to load your agent
agent = MainAgent()

print(f'Using agent {agent.__class__.__name__}, if this is incorrect please update the code to load in your agent')

file_name = str(inspect.getfile(CybORG))[:-10] + '/Evaluation/' + time.strftime("%Y%m%d_%H%M%S") + f'_{agent.__class__.__name__}.txt'
print(f'Saving evaluation results to {file_name}')
with open(file_name, 'a+') as data:
    data.write(f'CybORG v{cyborg_version}, {scenario}\n')
    data.write(f"wrappers: {wrap_line}\n")

path = str(inspect.getfile(CybORG))
path = path[:-10] + f'/Shared/Scenarios/{scenario}.yaml'

print(f'using CybORG v{cyborg_version}, {scenario}\n')
for green_agent in [False, True]:
    for num_steps in [30, 50, 100]:
        for red_agent in [B_lineAgent, RedMeanderAgent, SleepAgent]:

            if green_agent:
                cyborg = CybORG(path, 'sim', agents={'Red': red_agent, 'Green': GreenAgent})
            else:
                cyborg = CybORG(path, 'sim', agents={'Red': red_agent,})
            wrapped_cyborg = wrap(cyborg)

            observation, _ = wrapped_cyborg.reset()

            action_space = wrapped_cyborg.get_action_space(agent_name)
            total_reward = []
            actions = []
            latences = []
            for i in range(MAX_EPS):
                r = []
                a = []
                for j in range(num_steps):
                    start = time.time()
                    action = agent.get_action(observation, action_space)
                    observation, rew, done, info = wrapped_cyborg.step(action=action)
                    end = time.time()
                    r.append(rew)
                    a.append((str(cyborg.get_last_action('Blue')), str(cyborg.get_last_action('Red'))))
                    latences.append(end - start)

                agent.end_episode()
                total_reward.append(sum(r))
                actions.append(a)

                observation, _ = wrapped_cyborg.reset()

            iqm, bci, min_r, max_r = get_iqm_with_bci(total_reward)
            print(f"Results for {red_agent.__name__}, {num_steps} steps, {"with Green" if green_agent else "no Green"}")
            print(f"Reward IQM: {iqm:.2f} [{bci.confidence_interval.low:.2f}, {bci.confidence_interval.high:.2f}], min: {min_r:.2f}, max: {max_r:.2f}")
            print(f'Average time to take action: {mean(latences):.2e} s, +/- {stdev(latences):.2e}, min: {min(latences):.2e}, max: {max(latences):.2e}')
            print("")
            with open(file_name, 'a+') as data:
                data.write(f'steps: {num_steps}, adversary: {red_agent.__name__}, green: {green_agent}, iqm: {iqm:2f} [{bci.confidence_interval.low:2f}, {bci.confidence_interval.high:2f}], min: {min_r:2f}, max: {max_r:2f} \n')
                for act, sum_rew in zip(actions, total_reward):
                    data.write(f'actions: {act}, total reward: {sum_rew}\n')